<a href="https://colab.research.google.com/github/diegovc1987-boop/Ejercicio-4/blob/main/Ejercicio4_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install gradio transformers google-generativeai deep-translator

### Setup and API Key Configuration

We'll import the necessary libraries and configure the Gemini API key. Make sure your `GEMINI_API_KEY` is set in Colab's 'Secrets' panel (the key icon on the left sidebar).

In [2]:
import gradio as gr
from transformers import pipeline
import google.generativeai as genai
from deep_translator import GoogleTranslator
from google.colab import userdata
import os

# Configure Gemini API Key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") # Corrected: Removed os.getenv as userdata.get only takes one argument

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    # Using a faster Gemini model for quicker summarization
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
else:
    print("GOOGLE_API_KEY not found. Gemini summarization will not be available.")
    gemini_model = None

# Load HuggingFace sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Sentiment Analysis Function

This function uses a pre-trained HuggingFace model to determine the sentiment (positive or negative) of the input text.

In [3]:
def analyze_sentiment(text):
    if not text:
        return "Please enter some text."
    if not sentiment_pipeline:
        return "Sentiment analysis model not loaded."
    try:
        # Translate text to English before sentiment analysis
        translator = GoogleTranslator(source='auto', target='en')
        translated_text = translator.translate(text)

        result = sentiment_pipeline(translated_text[:512])[0]  # Truncate to 512 tokens
        sentiment = "POSITIVO 😊" if result["label"] == "POSITIVE" else "NEGATIVO 😞"
        score = round(result["score"], 4)
        return f"Sentiment: {sentiment} (Confidence: {score*100:.2f}%) - Original: '{text}', Translated: '{translated_text}'"
    except Exception as e:
        return f"Error in sentiment analysis: {e}"

sentiment_interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter text for sentiment analysis..."),
    outputs="text",
    title="📊 Sentiment Analyzer",
    description="Analyze the sentiment of your text (Positive or Negative)."
)

### Text Summarization Function

This function uses the Gemini model to generate a concise summary of the provided text.

In [4]:
def summarize_text(text):
    if not text:
        return "Please enter some text."
    if not gemini_model:
        return "Gemini model not configured. Please set GOOGLE_API_KEY in Colab secrets."

    MAX_SUMMARY_LENGTH = 10000 # Define a maximum length for summarization input
    original_text_length = len(text)
    if original_text_length > MAX_SUMMARY_LENGTH:
        print(f"Warning: Input text for summarization is too long ({original_text_length} chars). Truncating to {MAX_SUMMARY_LENGTH} chars.")
        text = text[:MAX_SUMMARY_LENGTH]

    try:
        prompt = f"""Resume the following text in Spanish concisely (max 3 sentences):

{text}"""
        print(f"[Summarizer] Processing text of length: {len(text)} for summarization.") # Diagnostic print
        print("[Summarizer] Calling gemini_model.generate_content...") # New diagnostic print
        response = gemini_model.generate_content(prompt)
        print("[Summarizer] Received response from gemini_model.generate_content.") # New diagnostic print
        return response.text
    except Exception as e:
        return f"Error in summarization: {e}"

summarization_interface = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(lines=7, placeholder="Enter text to summarize..."),
    outputs="text",
    title="📝 Text Summarizer",
    description="Get a concise summary of your text using Gemini."
)


### Translation Function

This function leverages `deep_translator` to translate text to a specified target language.

In [5]:
def translate_text(text, target_lang="es"):
    if not text:
        return "Please enter some text."
    try:
        translator = GoogleTranslator(source='auto', target=target_lang)
        result = translator.translate(text[:5000])  # Limit character count
        return result
    except Exception as e:
        return f"Error in translation: {e}"

translation_interface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter text to translate..."),
        gr.Dropdown(
            choices=["es", "en", "fr", "de", "it", "pt", "ja", "zh-CN"],
            label="Target Language",
            value="es"
        )
    ],
    outputs="text",
    title="🌍 Translator",
    description="Translate text to various languages."
)

### Launch the Gradio Multi-Tool Application

Finally, we combine all three interfaces into a single `gr.TabbedInterface` and launch the application. The public URL will be displayed below after execution.

In [6]:
multi_tool_app = gr.TabbedInterface(
    [sentiment_interface, summarization_interface, translation_interface],
    ["Sentiment Analyzer", "Text Summarizer", "Translator"],
    title="AI Multi-Tool (Gradio)"
)

multi_tool_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7e9d4336f03401fee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
